In [0]:
RAW_PATH = '/Volumes/workspace/default/ecommerce/'
BRONZE_PATH='/Volumes/workspace/default/ecommerce/bronze1/'
SILVER_PATH='/Volumes/workspace/default/ecommerce/silver1/'
GOLD_PATH='/Volumes/workspace/default/ecommerce/gold/'



In [0]:
silver = spark.read.format("Delta").load(SILVER_PATH+"transactions_enriched")

In [0]:
silver.createOrReplaceTempView("silver_txn")

In [0]:
spark.sql("""select * from silver_txn""")

DataFrame[customer_id: int, product_id: int, store_id: int, transaction_id: int, transaction_date: string, quantity: int, total_amount: double, customer_name: string, city: string, signup_date: date, product_name: string, category: string, price: double, brand: string, store_name: string, location: string, manager: string, opened_date: date, promotion_id: int, discount: int, start_date: date, end_date: date, channel: string, feedback_id: int, product_id_duplicate: int, rating: int, review: string, date: date, is_valid_store: boolean, has_valid_customer: boolean, promotion_active: boolean, final_amount: double]

In [0]:
gold_daily_store_cat=spark.sql("""
select 
  transaction_date,store_name,location,category,
  sum(final_amount) as net_sale,
  sum(total_amount) as gross_sale,
  count(*) as txn_count
  from silver_txn
  group by transaction_date,store_name,location,category
""")
display(gold_daily_store_cat)


transaction_date,store_name,location,category,net_sale,gross_sale,txn_count
2022-10-10,Store_4,Kolkata,Electronics,1010.4,1010.4,2
2024-02-16,Store_4,Kolkata,Electronics,2010.3000000000002,2073.3,3
2024-06-11,Store_2,Bangalore,Clothing,412.76,443.76,2
2023-12-26,Store_4,Kolkata,Furniture,3791.0,3907.0,4
2022-06-21,Store_1,Kolkata,Electronics,542.88,578.88,2
2022-12-13,Store_4,Kolkata,Furniture,1114.1,1172.1,2
2024-02-29,Store_3,Delhi,Electronics,3046.9500000000003,3109.9500000000003,3
2024-02-05,Store_4,Kolkata,Electronics,156.96,192.96,2
2023-07-07,Store_2,Bangalore,Furniture,723.4,781.4,2
2022-02-27,Store_2,Bangalore,Furniture,854.44,854.44,2


In [0]:
gold_daily_store_cat.write.format("delta").mode("overwrite").partitionBy('transaction_date').save(GOLD_PATH+"daily_store_category")

In [0]:
gold_top_customer = spark.sql(""" 
                              select customer_id,customer_name,store_name,
                              sum(final_amount) as total_spend,
                              count(*) as txn_count
                              from silver_txn
                              group by customer_id,customer_name,store_name
                              """
                              
                              )
display(gold_top_customer)


customer_id,customer_name,store_name,total_spend,txn_count
8,Customer_8,Store_4,1010.4,2
11,Customer_11,Store_4,3096.06,7
7,Customer_7,Store_2,1136.1599999999999,4
12,Customer_12,Store_4,3791.0,4
14,Customer_14,Store_1,542.88,2
9,Customer_9,Store_4,1114.1,2
9,Customer_9,Store_3,3782.7900000000004,5
10,Customer_10,Store_2,854.44,2
9,Customer_9,Store_1,202.08,1
7,Customer_7,Store_5,264.84,2


In [0]:
gold_top_customer.write.format('delta').mode('overwrite').save(GOLD_PATH+'top_customers')

In [0]:
gold_promo=spark.sql(""" 
                     select promotion_id,
                     count(*) as promo_txns,
                     sum(final_amount) as promo_sale,
                     avg(final_amount) as avg_promo_sale
                     from silver_txn
                     group by promotion_id
                     """)
display(gold_promo)


promotion_id,promo_txns,promo_sale,avg_promo_sale
null,9,4787.7300000000005,531.97
10,2,1681.75,840.875
3,3,641.6399999999999,213.87999999999997
9,4,2818.25,704.5625
8,6,1659.12,276.52
6,2,1697.75,848.875
2,3,596.6399999999999,198.87999999999997
1,4,2810.25,702.5625
7,6,1791.12,298.52
4,2,1677.75,838.875


In [0]:
gold_promo.write.format('delta').mode('overwrite').save(GOLD_PATH+'promo_impact')

In [0]:
gold_sentiment = spark.sql("""
                           select product_id,product_name, category,
                           avg(rating) as avg_rating,
                           count(rating) as rating_count
                           from silver_txn
                           group by product_id,product_name,category
                           """)
display(gold_sentiment)


product_id,product_name,category,avg_rating,rating_count
10,Product_10,Electronics,4.5,2
7,Product_7,Electronics,null,0
3,Product_3,Clothing,3.0,6
1,Product_1,Furniture,3.0,6
2,Product_2,Electronics,2.0,2
4,Product_4,Furniture,3.5,4
6,Product_6,Furniture,null,0


In [0]:
gold_sentiment.write.format('delta').mode('overwrite').save(GOLD_PATH+'product_sentiment')

In [0]:
from pyspark.sql.functions import max,count,sum,datediff,current_date,to_timestamp,when,col
rfm=silver.groupBy('customer_id').agg(max('transaction_date').alias('last_txn'),
                                      count('transaction_id').alias('frequency'),
                                      sum('final_amount').alias('monetary')).withColumn('recency_days',datediff(current_date(),'last_txn')  )

In [0]:
rfm=rfm.withColumn('recency_score',when(col('recency_days')<30,'0-30').when(col('recency_days')<90,'31-90').otherwise('90+')).withColumn("frequency_bascket",when(col('frequency')>=10,'High').when(col('frequency')>=3,'medium').otherwise('low')).withColumn('monetary_score',when(col('monetary')>=1000,'High').when(col('monetary')>=300,'medium').otherwise('low'))
display(rfm)

customer_id,last_txn,frequency,monetary,recency_days,recency_score,frequency_bascket,monetary_score
8,2022-10-10,2,1010.4,1477,90+,low,High
11,2024-02-16,7,3096.06,983,90+,medium,High
7,2024-06-11,6,1401.0,867,90+,medium,High
12,2023-12-26,4,3791.0,1035,90+,medium,High
14,2022-06-21,2,542.88,1588,90+,low,medium
9,2024-08-23,9,5526.1900000000005,794,90+,medium,High
10,2024-09-21,4,1708.88,765,90+,medium,High
5,2022-03-30,2,156.96,1671,90+,low,low
15,2024-08-14,3,2367.9500000000003,803,90+,medium,High
13,2023-11-10,2,560.68,1081,90+,low,medium


In [0]:
rfm.write.format('delta').mode('overwrite').save(GOLD_PATH+'rfm')